# 🤖 ArXiv Multi-Agent Research Pipeline
Mistral-7B-Instruct | RAG from ArXiv Papers

**Fixes in v3:**
-  Real citations injected (`Author et al., YEAR`) — no hallucinated `X et al.`
-  Clean header (no stray keyword strings)
-  Citation map built from retrieved papers and passed to every agent
-  LLM instructed to ONLY use provided citation keys — never invent references
-  Robust keyword cleaning (regex-based, never passes raw sentences)
-  Smart ArXiv query with 3-tier fallback

**Agents:** Planner → Research → Writer → Editor → Executor → Final Report

## 🔧 BLOCK 1 — Install Dependencies

In [ ]:
!pip install -q arxiv
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q "transformers>=4.40.0"
!pip install -q accelerate
!pip install -q bitsandbytes
!pip install -q einops

print('✅ All dependencies installed!')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.3 MB/s eta 0:00:00
✅ All dependencies installed!


## 📦 BLOCK 2 — Imports & User Config

In [ ]:
import arxiv
import faiss
import numpy as np
import torch
import re
import warnings
warnings.filterwarnings('ignore')

from dataclasses import dataclass
from typing import List, Dict, Any
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Imports done | Device: {DEVICE}')

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  ✏️  CHANGE THESE to customise your research run
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
TOPIC             = "Large Language Models for Code Generation"
MAX_ARXIV_RESULTS = 30    # papers to fetch from ArXiv
TOP_K_PAPERS      = 6     # papers to analyse deeply
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

✅ Imports done | Device: cuda


## 🧠 BLOCK 3 — Load Free LLM (Mistral-7B-Instruct, 4-bit)
> Runs on free Colab T4 GPU. No API key required.

In [ ]:
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
print(f'⏳ Loading {MODEL_ID}...')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

llm_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config if DEVICE == 'cuda' else None,
    device_map='auto' if DEVICE == 'cuda' else None,
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
    trust_remote_code=True
)
llm_model.eval()
print('✅ Mistral-7B-Instruct loaded!')


def generate_text(prompt: str,
                  max_new_tokens: int = 400,
                  system: str = "") -> str:
    """Mistral [INST] chat format. Returns only the newly generated text."""
    if system:
        full = f"<s>[INST] <<SYS>>\n{system}\n<</SYS>>\n\n{prompt} [/INST]"
    else:
        full = f"<s>[INST] {prompt} [/INST]"

    inputs = llm_tokenizer(
        full, return_tensors='pt', truncation=True, max_length=3072
    ).to(DEVICE)

    with torch.no_grad():
        out = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.65,
            top_p=0.9,
            repetition_penalty=1.15,
            pad_token_id=llm_tokenizer.eos_token_id
        )

    n_input = inputs['input_ids'].shape[1]
    return llm_tokenizer.decode(out[0][n_input:], skip_special_tokens=True).strip()


# Sanity test
print('LLM test:', generate_text(
    'What is code generation using LLMs? One sentence only.',
    max_new_tokens=60
))

⏳ Loading mistralai/Mistral-7B-Instruct-v0.2...


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

✅ Mistral-7B-Instruct loaded!
LLM test: Code generation using LLMs (Large Language Models) involves creating computer programs or scripts based on textual instructions generated by a large language model.


## 🔍 BLOCK 4 — RAG Pipeline (ArXiv + Embeddings + FAISS + Citation Map)

In [ ]:
print('⏳ Loading embedding model...')
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print('✅ Embedder ready!')


@dataclass
class Paper:
    title:      str
    authors:    List[str]
    abstract:   str
    url:        str
    published:  str
    categories: List[str]

    @property
    def cite_key(self) -> str:
        """
        Returns a clean in-text citation string like  'Smith et al. (2024)'
        so the LLM can use it directly in prose without inventing fake refs.
        """
        if self.authors:
            last = self.authors[0].split()[-1]   # last name of first author
        else:
            last = 'Unknown'
        year = self.published[:4]
        suffix = ' et al.' if len(self.authors) > 1 else ''
        return f"{last}{suffix} ({year})"


class ArXivRAGPipeline:
    """ArXiv fetch → embed → FAISS index → semantic retrieval."""

    def __init__(self):
        self.papers: List[Paper] = []
        self.index = None
        self._vecs = None

    # ── query builder ─────────────────────────────────────────────────
    @staticmethod
    def _build_query(keywords: List[str], topic: str) -> str:
        parts = [f'ti:"{topic}"']
        extras = [k for k in keywords if k.lower() not in topic.lower()][:2]
        for k in extras:
            parts.append(f'abs:"{k}"')
        return ' OR '.join(parts)

    # ── fetch ─────────────────────────────────────────────────────────
    def fetch_papers(self, keywords: List[str], topic: str,
                     max_results: int = 30) -> List[Paper]:

        def _search(q: str, n: int) -> List[Paper]:
            client = arxiv.Client(num_retries=3, page_size=50)
            search = arxiv.Search(query=q, max_results=n,
                                  sort_by=arxiv.SortCriterion.Relevance)
            out = []
            for r in client.results(search):
                out.append(Paper(
                    title      = r.title,
                    authors    = [a.name for a in r.authors[:4]],
                    abstract   = r.summary.replace('\n', ' ')[:1000],
                    url        = r.entry_id,
                    published  = str(r.published.date()),
                    categories = r.categories
                ))
            return out

        q1 = self._build_query(keywords, topic)
        print(f'🔍 ArXiv query (attempt 1): {q1}')
        papers = _search(q1, max_results)

        if len(papers) < 5:
            q2 = f'"{topic}"'
            print(f'⚠️  Sparse ({len(papers)}) — retry: {q2}')
            papers = _search(q2, max_results)

        if len(papers) < 5:
            q3 = ' OR '.join([f'"{k}"' for k in keywords[:3]])
            print(f'⚠️  Still sparse — broad retry: {q3}')
            papers = _search(q3, max_results)

        self.papers = papers
        print(f'✅ Retrieved {len(papers)} papers')
        return papers

    # ── index ─────────────────────────────────────────────────────────
    def build_index(self):
        assert self.papers, 'Fetch papers first.'
        texts = [f"{p.title}. {p.abstract}" for p in self.papers]
        print(f'⏳ Embedding {len(texts)} papers...')
        vecs = embedder.encode(texts, batch_size=16, show_progress_bar=True)
        vecs = np.array(vecs, dtype='float32')
        faiss.normalize_L2(vecs)
        dim = vecs.shape[1]
        self.index = faiss.IndexFlatIP(dim)
        self.index.add(vecs)
        self._vecs = vecs
        print(f'✅ FAISS index: {self.index.ntotal} vectors, dim={dim}')

    # ── retrieve ──────────────────────────────────────────────────────
    def retrieve(self, query: str, top_k: int = 6) -> List[Paper]:
        qv = embedder.encode([query], show_progress_bar=False).astype('float32')
        faiss.normalize_L2(qv)
        scores, idxs = self.index.search(qv, min(top_k, len(self.papers)))
        results = [
            self.papers[i] for s, i in zip(scores[0], idxs[0])
            if s >= 0.10 and i < len(self.papers)
        ]
        return results if results else self.papers[:top_k]

    # ── citation map ──────────────────────────────────────────────────
    @staticmethod
    def build_citation_map(papers: List[Paper]) -> Dict[int, str]:
        """
        Returns {1: 'Smith et al. (2024)', 2: ...} for the retrieved papers.
        Passed to all writing agents so they cite real papers, not hallucinations.
        """
        return {i + 1: p.cite_key for i, p in enumerate(papers)}


rag = ArXivRAGPipeline()
print('✅ RAG Pipeline ready!')

⏳ Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedder ready!
✅ RAG Pipeline ready!


## 📋 BLOCK 5 — Planner Agent

In [ ]:
class PlannerAgent:
    """
    Planner Agent
    -------------
    Input : topic string
    Output: plan dict  →  topic, keywords (clean list), research_questions, outline

    Keyword extraction uses regex parsing + bigram fallback so it
    NEVER passes a raw sentence to the ArXiv query.
    """

    SYS = ("You are a senior academic research planner. "
           "Always respond concisely and follow the exact format requested. "
           "Never add extra commentary.")

    def __init__(self, llm_fn):
        self.llm  = llm_fn
        self.name = '📋 Planner Agent'

    # ── keyword extraction ────────────────────────────────────────────
    def _keywords(self, topic: str) -> List[str]:
        prompt = (
            f'List exactly 5 academic search keywords for: "{topic}".\n'
            f'Respond with ONLY a comma-separated list. '
            f'No numbers, no bullets, no extra text.\n'
            f'Example format: keyword one, keyword two, keyword three, keyword four, keyword five'
        )
        raw = self.llm(prompt, max_new_tokens=60, system=self.SYS)

        # ── parse: keep only short, clean tokens ──
        # Strip anything that looks like a label (e.g. "Keywords:", "1.")
        raw = re.sub(r'^[\w\s]*:', '', raw).strip()
        candidates = [re.sub(r'[^\w\s\-]', '', k).strip()
                      for k in raw.split(',')]
        keywords = [
            k for k in candidates
            if 2 < len(k) < 55
            and not re.match(r'^\d', k)        # no numbered items
            and k.lower() not in ('none', 'na')
        ]

        # ── fallback: bigrams from topic ──
        if len(keywords) < 2:
            stop = {'for','the','a','an','of','in','on','with',
                    'and','or','to','using','via','by','from'}
            words = [w for w in topic.split() if w.lower() not in stop]
            bigrams = [f'{words[i]} {words[i+1]}' for i in range(len(words)-1)]
            keywords = (bigrams + words)[:5]

        return keywords[:5]

    # ── research questions ────────────────────────────────────────────
    def _questions(self, topic: str) -> str:
        prompt = (
            f'Write exactly 3 specific research questions for a survey on "{topic}".\n'
            f'Format (use exactly this):\n'
            f'1. <question>\n2. <question>\n3. <question>'
        )
        return self.llm(prompt, max_new_tokens=200, system=self.SYS)

    # ── outline ───────────────────────────────────────────────────────
    def _outline(self, topic: str) -> str:
        prompt = (
            f'Create a 5-section report outline for a survey on "{topic}".\n'
            f'Sections: Introduction, Key Concepts, Recent Advances, '
            f'Challenges, Conclusion.\n'
            f'One sentence per section describing its content.'
        )
        return self.llm(prompt, max_new_tokens=200, system=self.SYS)

    # ── public ────────────────────────────────────────────────────────
    def run(self, topic: str) -> Dict[str, Any]:
        print(f'\n{self.name} running...')
        kws       = self._keywords(topic)
        questions = self._questions(topic)
        outline   = self._outline(topic)
        plan = dict(topic=topic, keywords=kws,
                    research_questions=questions, outline=outline)
        print(f'✅ Plan ready | keywords: {kws}')
        return plan


planner_agent = PlannerAgent(generate_text)
print('✅ Planner Agent ready!')

✅ Planner Agent ready!


## 🔍 BLOCK 6 — Research Agent

In [ ]:
class ResearchAgent:
    """
    Research Agent
    --------------
    1. Fetches papers from ArXiv using plan keywords.
    2. Builds FAISS index.
    3. Retrieves top-k semantically relevant papers.
    4. Summarises each paper (problem / method / result).
    5. Builds a citation_map  {1: 'Smith et al. (2024)', ...}
       that all downstream agents use for real citations.
    """

    SYS = ("You are a careful academic researcher. "
           "Summarise papers accurately and extract key technical contributions. "
           "Be factual and concise.")

    def __init__(self, rag_pipeline: ArXivRAGPipeline, llm_fn):
        self.rag  = rag_pipeline
        self.llm  = llm_fn
        self.name = '🔍 Research Agent'

    def _summarise(self, paper: Paper) -> str:
        prompt = (
            f'Summarise this paper in exactly 3 sentences:\n'
            f'Sentence 1 — what problem it addresses.\n'
            f'Sentence 2 — the method or approach used.\n'
            f'Sentence 3 — the main result or contribution.\n\n'
            f'Title: {paper.title}\n'
            f'Abstract: {paper.abstract[:700]}'
        )
        return self.llm(prompt, max_new_tokens=180, system=self.SYS)

    def run(self, plan: Dict[str, Any],
            max_results: int = 30, top_k: int = 6) -> Dict[str, Any]:
        print(f'\n{self.name} running...')

        # 1 fetch
        all_papers = self.rag.fetch_papers(
            plan['keywords'], plan['topic'], max_results)

        # 2 index
        self.rag.build_index()

        # 3 retrieve
        relevant = self.rag.retrieve(plan['topic'], top_k=top_k)
        print(f'📎 Top {len(relevant)} papers selected:')
        for i, p in enumerate(relevant):
            print(f'   [{i+1}] {p.title[:72]}')

        # 4 summarise
        summaries = []
        for i, p in enumerate(relevant):
            print(f'   Summarising [{i+1}/{len(relevant)}] ...')
            summaries.append({'paper': p, 'summary': self._summarise(p)})

        # 5 build citation map  →  {1: 'Huynh et al. (2025)', ...}
        citation_map = ArXivRAGPipeline.build_citation_map(relevant)

        print(f'✅ {self.name} done!')
        return dict(
            all_papers   = all_papers,
            relevant     = relevant,
            summaries    = summaries,
            citation_map = citation_map
        )


research_agent = ResearchAgent(rag, generate_text)
print('✅ Research Agent ready!')

✅ Research Agent ready!


## ✍️ BLOCK 7 — Writer Agent
> Injects the real citation map into every prompt so Mistral cites actual papers.

In [ ]:
class WriterAgent:
    """
    Writer Agent
    ------------
    Writes each section using:
      - numbered paper summaries as factual context
      - explicit citation keys (e.g. 'Huynh et al. (2025)') injected into the prompt
      - a hard instruction: ONLY use the provided citation keys, never invent new ones
    """

    SYS = (
        "You are an expert academic writer composing a research survey. "
        "Write in a clear, formal, third-person academic style. "
        "IMPORTANT: When citing papers, use ONLY the exact citation keys "
        "provided in the prompt (e.g. Smith et al. (2024)). "
        "Do NOT invent any citations like 'X et al.' or 'Authors (2025)'. "
        "If unsure, write the claim without a citation rather than fabricating one."
    )

    SECTION_GUIDE = {
        'Introduction': (
            'Write 2 paragraphs.\n'
            'Para 1: Motivate the research area and explain why it matters.\n'
            'Para 2: Describe the scope and structure of this survey.'
        ),
        'Key Concepts': (
            'Write 2 paragraphs.\n'
            'Para 1: Define the 3–4 most important technical terms appearing across the papers.\n'
            'Para 2: Explain the foundational methods and architectural ideas used in this field.'
        ),
        'Recent Advances': (
            'Write 3 paragraphs.\n'
            'Para 1: Describe notable model or technique contributions from the papers.\n'
            'Para 2: Discuss specific benchmarks, datasets, or evaluation frameworks introduced.\n'
            'Para 3: Compare different approaches and highlight convergent trends.'
        ),
        'Challenges': (
            'Write 2 paragraphs.\n'
            'Para 1: Identify limitations and open problems mentioned across the papers.\n'
            'Para 2: Highlight gaps that future research should address.'
        ),
        'Conclusion': (
            'Write 2 paragraphs.\n'
            'Para 1: Summarise the key takeaways from all reviewed papers.\n'
            'Para 2: Suggest the most promising directions for future work.'
        )
    }

    def __init__(self, llm_fn):
        self.llm  = llm_fn
        self.name = '✍️ Writer Agent'

    # ── helpers ───────────────────────────────────────────────────────
    @staticmethod
    def _context_block(summaries: List[Dict],
                       citation_map: Dict[int, str]) -> str:
        """Numbered paper summaries with their cite key shown explicitly."""
        lines = []
        for i, s in enumerate(summaries):
            p    = s['paper']
            ckey = citation_map.get(i + 1, p.cite_key)
            lines.append(
                f"[{i+1}] CITE AS: {ckey}\n"
                f"    Title: {p.title}\n"
                f"    Summary: {s['summary']}"
            )
        return '\n\n'.join(lines)

    @staticmethod
    def _cite_keys_list(citation_map: Dict[int, str]) -> str:
        return ', '.join(citation_map.values())

    def _write_section(self, section: str, context: str,
                       topic: str, cite_keys: str) -> str:
        guide = self.SECTION_GUIDE.get(section, f'Write the {section} section.')
        prompt = (
            f'Topic: "{topic}"\n\n'
            f'AVAILABLE CITATION KEYS (use ONLY these, no others):\n'
            f'{cite_keys}\n\n'
            f'PAPER SUMMARIES (your factual source):\n{context}\n\n'
            f'TASK — {section} section:\n{guide}\n\n'
            f'Write the {section} section now (formal academic prose):'
        )
        return self.llm(prompt, max_new_tokens=480, system=self.SYS)

    # ── public ────────────────────────────────────────────────────────
    def run(self, plan: Dict[str, Any],
            research: Dict[str, Any]) -> Dict[str, Any]:
        print(f'\n{self.name} running...')

        topic        = plan['topic']
        citation_map = research['citation_map']
        context      = self._context_block(research['summaries'], citation_map)
        cite_keys    = self._cite_keys_list(citation_map)
        sections     = {}

        for sec in self.SECTION_GUIDE:
            print(f'   Writing: {sec} ...')
            sections[sec] = self._write_section(sec, context, topic, cite_keys)

        references = [
            f"[{i+1}] {s['paper'].authors[0] if s['paper'].authors else 'Unknown'} "
            f"et al. ({s['paper'].published[:4]}). "
            f"{s['paper'].title}. {s['paper'].url}"
            for i, s in enumerate(research['summaries'])
        ]

        print(f'✅ {self.name} done!')
        return dict(
            topic       = topic,
            sections    = sections,
            references  = references,
            papers_used = len(research['summaries'])
        )


writer_agent = WriterAgent(generate_text)
print('✅ Writer Agent ready!')

✅ Writer Agent ready!


## 📝 BLOCK 8 — Editor Agent

In [ ]:
class EditorAgent:
    """
    Editor Agent
    ------------
    Improves each section for clarity, flow, and academic rigour.
    Critically: it is instructed NOT to replace real citations with fake ones.
    Falls back to original content if edited version is suspiciously short.
    """

    SYS = (
        "You are a professional academic editor. "
        "Improve text for clarity, flow, and academic tone. "
        "Do NOT change factual content. "
        "Do NOT replace or remove existing citations. "
        "Do NOT add new citations that were not in the original text."
    )

    def __init__(self, llm_fn):
        self.llm  = llm_fn
        self.name = '📝 Editor Agent'

    def _edit(self, name: str, content: str) -> str:
        prompt = (
            f'Edit the "{name}" section below for:\n'
            f'1. Sentence clarity and academic tone\n'
            f'2. Removing redundant sentences\n'
            f'3. Smooth paragraph transitions\n'
            f'4. Keeping all facts and citations intact\n\n'
            f'--- ORIGINAL ---\n{content}\n--- END ---\n\n'
            f'Write the improved version:'
        )
        edited = self.llm(prompt, max_new_tokens=520, system=self.SYS)
        # Safety: discard if too short (LLM returned garbage)
        return edited if len(edited) > max(80, len(content) * 0.4) else content

    def run(self, draft: Dict[str, Any]) -> Dict[str, Any]:
        print(f'\n{self.name} running...')
        edited = {}
        for name, content in draft['sections'].items():
            print(f'   Editing: {name} ...')
            edited[name] = self._edit(name, content)
        print(f'✅ {self.name} done!')
        return {**draft, 'sections': edited, 'edited': True}


editor_agent = EditorAgent(generate_text)
print('✅ Editor Agent ready!')

✅ Editor Agent ready!


## ⚙️ BLOCK 9 — Executor Agent

In [ ]:
class ExecutorAgent:
    """
    Executor Agent
    --------------
    Orchestrates the full pipeline:
      Planner → Research → Writer → Editor
    Maintains shared state and reports progress.
    """

    def __init__(self, planner, researcher, writer, editor):
        self.planner    = planner
        self.researcher = researcher
        self.writer     = writer
        self.editor     = editor
        self.name       = '⚙️  Executor Agent'
        self.state: Dict[str, Any] = {}

    def run(self, topic: str,
            max_results: int = 30, top_k: int = 6) -> Dict[str, Any]:
        SEP = '═' * 65
        print(f'\n{SEP}')
        print(f'  {self.name}')
        print(f'  Topic : {topic}')
        print(f'{SEP}\n')

        steps = [
            ('1/4  Planning',       'plan',         lambda: self.planner.run(topic)),
            ('2/4  Researching',    'research',     lambda: self.researcher.run(
                                                        self.state['plan'],
                                                        max_results=max_results,
                                                        top_k=top_k)),
            ('3/4  Writing draft',  'draft',        lambda: self.writer.run(
                                                        self.state['plan'],
                                                        self.state['research'])),
            ('4/4  Editing draft',  'edited_draft', lambda: self.editor.run(
                                                        self.state['draft'])),
        ]

        for label, key, fn in steps:
            try:
                print(f'🔄 STEP {label}...')
                self.state[key] = fn()
            except Exception as e:
                print(f'❌ Failed at step {label}: {e}')
                raise

        print(f'\n{SEP}')
        print(f'  ✅  Pipeline complete!')
        print(f'{SEP}\n')
        return self.state


executor_agent = ExecutorAgent(
    planner_agent, research_agent, writer_agent, editor_agent
)
print('✅ Executor Agent ready!')

✅ Executor Agent ready!


## 📄 BLOCK 10 — Final Report Agent

In [ ]:
class FinalReportAgent:
    """
    Final Report Agent
    ------------------
    - Generates an executive summary from Introduction + Conclusion.
    - Assembles all sections into a clean, formatted report.
    - Header shows only clean metadata (no stray LLM output).
    - Saves report to disk and returns it as a string.
    """

    SYS = ("You are an expert at writing concise executive summaries "
           "for academic research surveys. Be factual and specific.")

    def __init__(self, llm_fn):
        self.llm  = llm_fn
        self.name = '📄 Final Report Agent'

    def _exec_summary(self, topic: str,
                      sections: Dict[str, str],
                      cite_keys: str) -> str:
        ctx = (
            sections.get('Introduction', '')[:350] + ' ' +
            sections.get('Conclusion',   '')[:350]
        )
        prompt = (
            f'Write a 3-sentence executive summary for a survey on "{topic}".\n'
            f'Cover: (1) importance of the field, '
            f'(2) key findings, (3) future directions.\n'
            f'Use ONLY these citation keys if citing: {cite_keys}\n'
            f'Context:\n{ctx}'
        )
        return self.llm(prompt, max_new_tokens=200, system=self.SYS)

    def run(self, state: Dict[str, Any]) -> str:
        print(f'\n{self.name} running...')

        plan         = state['plan']
        research     = state['research']
        edited_draft = state['edited_draft']

        topic        = plan['topic']
        sections     = edited_draft['sections']
        references   = edited_draft['references']
        papers_used  = edited_draft['papers_used']
        citation_map = research['citation_map']
        cite_keys    = ', '.join(citation_map.values())

        print('   Generating executive summary...')
        exec_summary = self._exec_summary(topic, sections, cite_keys)

        # ── assemble ────────────────────────────────────────────────────
        SEP  = '=' * 70
        THIN = '-' * 70

        lines = [
            SEP,
            f'  RESEARCH REPORT: {topic.upper()}',
            SEP,
            f'  Papers retrieved : {len(research["all_papers"])}',
            f'  Papers analysed  : {papers_used}',
            # ← clean list, not a raw LLM sentence
            f'  Keywords used    : {", ".join(plan["keywords"])}',
            SEP,
            '',
            '📌 RESEARCH QUESTIONS',
            THIN,
            plan['research_questions'].strip(),
            '',
            '📌 EXECUTIVE SUMMARY',
            THIN,
            exec_summary.strip(),
            '',
        ]

        for sec_name, content in sections.items():
            lines += [
                f'📖 {sec_name.upper()}',
                THIN,
                content.strip(),
                ''
            ]

        lines += ['📚 REFERENCES', THIN]
        lines += references
        lines += ['', SEP, '  END OF REPORT', SEP]

        report = '\n'.join(lines)

        # ── save ────────────────────────────────────────────────────────
        safe = re.sub(r'[^\w\s-]', '', topic).replace(' ', '_')[:50]
        fname = f'report_{safe}.txt'
        with open(fname, 'w', encoding='utf-8') as f:
            f.write(report)

        print(f'✅ Report saved → {fname}')
        return report


final_report_agent = FinalReportAgent(generate_text)
print('✅ Final Report Agent ready!')

✅ Final Report Agent ready!


## 🚀 BLOCK 11 — RUN THE FULL PIPELINE
> Set `TOPIC`, `MAX_ARXIV_RESULTS`, `TOP_K_PAPERS` in **Block 2**, then run this block.

In [ ]:
# ── Run all agents ──────────────────────────────────────────────────
state = executor_agent.run(
    topic       = TOPIC,
    max_results = MAX_ARXIV_RESULTS,
    top_k       = TOP_K_PAPERS
)

# ── Produce final report ────────────────────────────────────────────
final_report = final_report_agent.run(state)

print('\n\n')
print(final_report)


═════════════════════════════════════════════════════════════════
  ⚙️  Executor Agent
  Topic : Large Language Models for Code Generation
═════════════════════════════════════════════════════════════════

🔄 STEP 1/4  Planning...

📋 Planner Agent running...
✅ Plan ready | keywords: ['Large scale models', 'Neural code generation', 'Machine learning programming', 'Natural language processing', 'Code synthesis']
🔄 STEP 2/4  Researching...

🔍 Research Agent running...
🔍 ArXiv query (attempt 1): ti:"Large Language Models for Code Generation" OR abs:"Large scale models" OR abs:"Neural code generation"
✅ Retrieved 30 papers
⏳ Embedding 30 papers...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

✅ FAISS index: 30 vectors, dim=384
📎 Top 6 papers selected:
   [1] Large Language Models for Code Generation: A Comprehensive Survey of Cha
   [2] A Survey on Evaluating Large Language Models in Code Generation Tasks
   [3] A Survey on Large Language Models for Code Generation
   [4] CodeVisionary: An Agent-based Framework for Evaluating Large Language Mo
   [5] Large Language Models for Code Generation: The Practitioners Perspective
   [6] Beyond Autoregression: An Empirical Study of Diffusion Large Language Mo
   Summarising [1/6] ...
   Summarising [2/6] ...
   Summarising [3/6] ...
   Summarising [4/6] ...
   Summarising [5/6] ...
   Summarising [6/6] ...
✅ 🔍 Research Agent done!
🔄 STEP 3/4  Writing draft...

✍️ Writer Agent running...
   Writing: Introduction ...
   Writing: Key Concepts ...
   Writing: Recent Advances ...
   Writing: Challenges ...
   Writing: Conclusion ...
✅ ✍️ Writer Agent done!
🔄 STEP 4/4  Editing draft...

📝 Editor Agent running...
   Editing: Introduction .

## 🗂️ BLOCK 12 — Browse Retrieved Papers

In [ ]:
all_p = state['research']['all_papers']
top_p = state['research']['relevant']

print(f'📄 ALL {len(all_p)} RETRIEVED PAPERS')
print('-' * 65)
for i, p in enumerate(all_p):
    print(f'[{i+1:02d}] {p.title}')
    print(f'     Authors   : {", ".join(p.authors)}')
    print(f'     Published : {p.published}  |  Cite: {p.cite_key}')
    print(f'     URL       : {p.url}')
    print(f'     Abstract  : {p.abstract[:200]}...\n')

print('\n' + '-' * 65)
print(f'📎 TOP {len(top_p)} PAPERS ANALYSED (by semantic similarity):')
print('-' * 65)
for i, s in enumerate(state['research']['summaries']):
    p = s['paper']
    print(f'[{i+1}] {p.title}')
    print(f'    Cite as  : {p.cite_key}')
    print(f'    Summary  : {s["summary"]}\n')

📄 ALL 30 RETRIEVED PAPERS
-----------------------------------------------------------------
[01] Planning with Large Language Models for Code Generation
     Authors   : Shun Zhang, Zhenfang Chen, Yikang Shen, Mingyu Ding
     Published : 2023-03-09  |  Cite: Zhang et al. (2023)
     URL       : http://arxiv.org/abs/2303.05510v1
     Abstract  : Existing large language model-based code generation pipelines typically use beam search or sampling algorithms during the decoding process. Although the programs they generate achieve high token-match...

[02] A Survey on Large Language Models for Code Generation
     Authors   : Juyong Jiang, Fan Wang, Jiasi Shen, Sungju Kim
     Published : 2024-06-01  |  Cite: Jiang et al. (2024)
     URL       : http://arxiv.org/abs/2406.00515v2
     Abstract  : Large Language Models (LLMs) have garnered remarkable advancements across diverse code-related tasks, known as Code LLMs, particularly in code generation that generates source code with LLM from nat

## 💾 BLOCK 13 — Download Report

In [ ]:
from google.colab import files
import glob

report_files = sorted(glob.glob('report_*.txt'))
if report_files:
    print(f'📥 Downloading: {report_files[-1]}')
    files.download(report_files[-1])
else:
    print('No report file found — run Block 11 first.')

📥 Downloading: report_Large_Language_Models_for_Code_Generation.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>